# CLIP Clustering

This notebook walks you through the code that was experimented with to cluster the glashelder dataset using CLIP embeddings.

The main technologies being used are:
- [CLIP pretrained model](https://huggingface.co/laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K)
- [UMAP dimensionality reduction algorithm](https://umap-learn.readthedocs.io/en/latest/)
- [HDBSCAN density based spacial clustering algorithm for applications with noise](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html)

The data flow of the code in this notebook looks like this:

```mermaid
flowchart LR
  i[images]@{shape: procs} -->|N images| c[CLIP]
  c -->|embedding 768xN| u1["UMAP(1)"]
  u1 -->|reduced 30xN| h
  u1 -->|reduced 30xN| u2["UMAP(2)"]
  u2 -->|reduced 3xN| v
  h[HDBSCAN] --> clusters@{shape: procs}
  h --> noise
  noise --> v
  clusters --> v
```

CLIP generates embeddings in 768 dimensions, which is too high for any clustering algorithm to process efficiently. \
Therefore, UMAP is used to reduce the dimensionality partially, to 30 dimensions. \
The lower the dimensionality in which the clustering takes place, the faster, but going too low may 'squish' multiple distinct spaces onto eachother.

For visualizing the clusters, a second UMAP dimensionality reduction is used, to get the data visualizable in 3D.

# Setup

### Dependencies

In [ ]:
import glob

from hdbscan import HDBSCAN
from umap import UMAP

from cluster.embed import get_embedder
from cluster.rr import log_clustered_points, log_images_3d
from common import PROJECT_ROOT
from common.clipmodel import get_device

### Configs
- `MODEL`: the CLIP model to get from huggingface
- `PACKAGE_ROOT`: filesystem path of the directory with files related to the `cluster` experiment
- `EMBEDDINGS_PATH`: the location where the embeddings will be saved to in a file

In [3]:
MODEL = "laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K"
PACKAGE_ROOT = PROJECT_ROOT / "cluster"
EMBEDDINGS_PATH = PACKAGE_ROOT / "embeddings.pkl"

**`IMAGE_DIR`: you need to modify this to be the path where the glashelder images are stored**

In [4]:
IMAGE_DIR = "/mnt/share/shares/glashelder/assets-crop"  # MODIFY ME

Downloading the model and setting it up for your device (cuda, cpu, mps)

In [ ]:
device = get_device()
embedder = get_embedder("clip")
embedder.load(device)

## Make image embeddings

In [6]:
# this makes a 'random' (images have random order) selection of images in the dataset of limited size,
# for speed. If you want a larger dataset to look through, increase this variable, `count`:
count = 4000
image_paths = sorted(glob.glob(f"{IMAGE_DIR}/*.jpg"))[:count]

*generate the embeddings of the images*

In [ ]:
embeddings, image_paths = embedder.encode_images(image_paths, batch_size=64)
embeddings.shape  # count vectors of length 768

# Reduce embeddings' dimensionality

In [8]:
# 30 dimensions for clustering
umap_model_30 = UMAP(n_components=30, n_neighbors=15, min_dist=0.20)
embeddings_30 = umap_model_30.fit_transform(embeddings)

In [9]:
# 3 dimensions for visualization
umap_model_3 = UMAP(n_components=3, n_neighbors=15, min_dist=0.20)
embeddings_3 = umap_model_30.fit_transform(embeddings_30)

# Cluster in 30 dimensions

In [10]:
clusterer = HDBSCAN(min_cluster_size=10)
clusterer.fit(embeddings_30)

/workspace/glashelder/.venv/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/workspace/glashelder/.venv/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


,min_cluster_size,10
,min_samples,None
,cluster_selection_epsilon,0.0
,cluster_selection_persistence,0.0
,max_cluster_size,0
,metric,'euclidean'
,alpha,1.0
,p,None
,algorithm,'best'
,leaf_size,40
,memory,Memory(location=None)


In [11]:
clusterer.labels_

array([1, 1, 1, ..., 1, 1, 1], shape=(4000,))

# Show clusters
Note: chances are high that the clustering doesn't return desired results when you supply it a limited number of images. You can mitigate this by increasing the `count` variable at the top half of this notebook.

In [12]:
import rerun as rr

In [37]:
rr.init("cluster_notebook")
rr.save("clusters.rrd")
log_clustered_points(embeddings_3, clusterer.labels_, count=count)
log_images_3d(embeddings_3, image_paths, 100)

HTML(value='<div id="4127c431-028d-4487-ba5c-079a93309ee8"><style onload="eval(atob(\'KGFzeW5jIGZ1bmN0aW9uICgp…

Open the saved file 'clusters.rrd' in the rerun viewer (e.g. by first downloading it from the jupyter UI)